In [83]:
# Import libraries
import pandas as pd
import numpy as np

# Load table
df = pd.read_excel('Day36_GroupBy_Aggregation.xlsx', sheet_name = 'Raw Data', header = 1)

# Revenue Calculation
df['revenue'] = (df['unit_price'] * df['quantity'] * (1 - df['discount'])).round(2)

In [84]:
# Add order_month
df['order_date'] = pd.to_datetime(df['order_date'])
df['order_month'] = df['order_date'].dt.month
df['order_quarter'] = df['order_date'].dt.quarter

In [85]:
# Goal: find total revenue per region
# Method: groupby single column, sum revenue, sort descending
result_a1 = df.groupby('region')['revenue'].sum().sort_values(ascending=False)
print(result_a1)

region
West     200275.0
North    162295.0
East     130100.0
South     95460.0
Name: revenue, dtype: float64


#### West - ₹200,275(Highest Revenue)
#### North - ₹162,295
#### East - ₹130,100
#### South - ₹95,460(Lowest Revenue)
#### Action: Condict a comparatative analysis of the sales tactics and product demand in the West to identify best practices and them implement a targeted marketing campaign in South to bridge the nearly 50% revenue gap.

In [86]:
# Goal: find average revenue per salesperson
# Method: group by salesperson, calculate mean revenue, round to 2 decimal places
result_a2 = df.groupby('salesperson')['revenue'].mean().round(2).sort_values(ascending = False)
print(result_a2)
print(f"Highest: {result_a2.idxmax()} with an average of ₹{result_a2.max():,.2f}")

salesperson
Aarav Shah     27049.17
Karan Mehta    26020.00
Priya Joshi    22707.50
Divya Rao      21889.00
Meera Nair     19092.00
Name: revenue, dtype: float64
Highest: Aarav Shah with an average of ₹27,049.17


In [87]:
# Goal: find total quantity sold per category
# Method: group by category, sum quantity, sort desending
result_a3 = df.groupby('category')['quantity'].sum().sort_values(ascending = False)
print(result_a3)
print(f"Highest Units: {result_a3.idxmax()} with most units of {result_a3.max()}")

category
Clothing          27
Home & Kitchen    21
Electronics       19
Name: quantity, dtype: int64
Highest Units: Clothing with most units of 27


In [88]:
# Goal : count order_id per region
# Method: group by region, count order_id, sort descending
result_a4 = df.groupby('region')['order_id'].count().sort_values(ascending=False)
print(result_a4)

region
West     9
North    6
East     5
South    5
Name: order_id, dtype: int64


#### Yes the order count is proportional to revenue.In revenue and order count the West region ranks first followed by North, then East and South.

#### Not necessarily North's lead doesn't automatically mean their salesperson are better it could be due to having a larger team, more orders or focus on expensive items like Electronics.To find the real cause I would check average revenue per salesperson and revenue per category to see the performance is consistent across region.

In [89]:
# Goal: total revenue by region and category
# Method: Group by two columns, sum revenue, then unstack to create a table
result_b1 = df.groupby(['region','category'])['revenue'].sum().unstack()
print(result_b1)

category  Clothing  Electronics  Home & Kitchen
region                                         
East       17300.0     107400.0          5400.0
North       4275.0     140620.0         17400.0
South      12900.0      74200.0          8360.0
West       23600.0     159700.0         16975.0


In [90]:
# Goal: find the highest combination revenue
# Method: Use the stack and idxmax
print(result_b1.stack().idxmax())

('West', 'Electronics')


In [91]:
# Goal: total revenue by salesperson and category
# Method: Group by two columns, sum revenue
result_b3 = df.groupby(['salesperson','category'])['revenue'].sum().unstack()
print(result_b3)
strongest = result_b3['Electronics'].idxmax()
print(f"Stongest in Electronics: {strongest}")

category     Clothing  Electronics  Home & Kitchen
salesperson                                       
Aarav Shah     4275.0     140620.0         17400.0
Divya Rao      7920.0      84550.0         16975.0
Karan Mehta   17300.0     107400.0          5400.0
Meera Nair    12900.0      74200.0          8360.0
Priya Joshi   15680.0      75150.0             NaN
Stongest in Electronics: Aarav Shah


#### unstack() pivots the innermost index level from rows into horizontal column headers.
#### It is more readable because it transforms a long repetitive vertical list into clean matrix style table.This allows to compare values across categories and regions at single glance..

In [92]:
# Goal: total revenue, avg revenue, order count, max order by salesperson
# Method: Group by salesperson and calculate sum,mean,count and max on revenue
result_c1 = df.groupby('salesperson')['revenue'].agg(['sum', 'mean', 'count', 'max'])
print(result_c1)

                  sum          mean  count      max
salesperson                                        
Aarav Shah   162295.0  27049.166667      6  85500.0
Divya Rao    109445.0  21889.000000      5  42750.0
Karan Mehta  130100.0  26020.000000      5  66000.0
Meera Nair    95460.0  19092.000000      5  41800.0
Priya Joshi   90830.0  22707.500000      4  42750.0


In [93]:
# Goal: rename the columns and make final sorted table of total revenue
# Method: Rename the colums , sort by total revenue, sort descending
result_c2 = result_c1.rename(columns={
    'sum': 'total_revenue',
    'mean': 'avg_revenue',
    'count': 'order_count',
    'max': 'max_order'
})

result_c2 = result_c2.sort_values(by = 'total_revenue', ascending=False)
print(result_c2)

             total_revenue   avg_revenue  order_count  max_order
salesperson                                                     
Aarav Shah        162295.0  27049.166667            6    85500.0
Karan Mehta       130100.0  26020.000000            5    66000.0
Divya Rao         109445.0  21889.000000            5    42750.0
Meera Nair         95460.0  19092.000000            5    41800.0
Priya Joshi        90830.0  22707.500000            4    42750.0


In [101]:
# Goal: Which category is the best revenue per unit
# Method: Group by category and apply multiple aggregations
result_c3 = df.groupby('category').agg({
    'quantity':'sum',
    'revenue':['sum', 'mean']
})
print(result_c3)

               quantity   revenue              
                    sum       sum          mean
category                                       
Clothing             27   58075.0   7259.375000
Electronics          19  481920.0  43810.909091
Home & Kitchen       21   48135.0   8022.500000


In [95]:
# Goal: Identify the category with highest revenue efficiency
# Method: Add a rev_per_unit column and sort the data
result_c3['rev_per_unit'] = result_c3[('revenue', 'sum')]/result_c3[('quantity','sum')]
result_c4 = result_c3.sort_values(by = 'rev_per_unit', ascending = False)
print(result_c4)
best_cat = result_c4.index[0]
val = result_c4['rev_per_unit'].iloc[0]
print(f"Category with best revenue per unit: {best_cat} (₹{val:,.2f})")

               quantity   revenue                rev_per_unit
                    sum       sum          mean              
category                                                     
Electronics          19  481920.0  43810.909091  25364.210526
Home & Kitchen       21   48135.0   8022.500000   2292.142857
Clothing             27   58075.0   7259.375000   2150.925926
Category with best revenue per unit: Electronics (₹25,364.21)


#### Total revenue is a great starting point but it doesn't tell the whole story.While Aarav Shah leads in total earnings you should also look a order_count for consistency and avg_revenue to see the actual of their average deal.For instance a salesperson might have high revenue simply because of one lucky large order(max_order) where as someone with high order count but lower total might be more reliable and consistent in long run.

In [96]:
# Goal : Monthly revenue trend
# Method: Group by order_month by revenue
result_d1 = df.groupby('order_month')['revenue'].sum()
print(result_d1)

order_month
1    112900.0
2     62375.0
3    125030.0
4     86800.0
5     85575.0
6     64600.0
7     50850.0
Name: revenue, dtype: float64


In [97]:
# Goal: Calculate the percentage growth month over month
#Method: Use .pct_change() on result_d1 and multiply by 100 for percentage
result_d2 = (result_d1.pct_change()*100).round(2)
print(result_d2)

order_month
1       NaN
2    -44.75
3    100.45
4    -30.58
5     -1.41
6    -24.51
7    -21.28
Name: revenue, dtype: float64


In [98]:
# Goal: Calculate quarterly revenue by category
# Method: groupby order_quarter and category, sum revenue, unstack for readability
result_d3 = df.groupby(['order_quarter','category'])['revenue'].sum().unstack()
print(result_d3)
top_combo = result_d3.stack().idxmax()
top_value = result_d3.stack().max()
print(f"The dominating combination is {top_combo} with ₹{top_value:,.2f}")

category       Clothing  Electronics  Home & Kitchen
order_quarter                                       
1               22520.0     248650.0         29135.0
2               27455.0     190520.0         19000.0
3                8100.0      42750.0             NaN
The dominating combination is (np.int32(1), 'Electronics') with ₹248,650.00


#### No it is not true.According to results_d2 output revenue declined in Month 2(-44.75%), Month4(-30.58%), Month 5(-1.41%), Month 6(-24.51%) and Month 7(-21.28%)

#### While raw totals only show the absolute volume of sales, pct_change reveals momentum and direction of the business.It hoghlights specific periods of decline(negative growth) that might be missed if you only look at large total numbers.

In [99]:
# Goal: Build final performance table
# Method: Use C2 result and add top category logic
e1_table = result_c2[['total_revenue', 'order_count', 'avg_revenue']].copy()
e1_table['top_category'] = df.groupby(['salesperson', 'category'])['revenue'].sum().unstack().idxmax(axis=1)
print(e1_table)

             total_revenue  order_count   avg_revenue top_category
salesperson                                                       
Aarav Shah        162295.0            6  27049.166667  Electronics
Karan Mehta       130100.0            5  26020.000000  Electronics
Divya Rao         109445.0            5  21889.000000  Electronics
Meera Nair         95460.0            5  19092.000000  Electronics
Priya Joshi        90830.0            4  22707.500000  Electronics


In [100]:
# Goal: find the single best region-salesperson pair
# Method: Group by both columns, sum revenue and count orders
pair_stats = df.groupby(['region', 'salesperson'])['revenue'].agg(['sum', 'count'])
best_pair = pair_stats['sum'].idxmax()
top_revenue = pair_stats['sum'].max()
order_num = pair_stats.loc[best_pair, 'count']

print(f"Best: {best_pair[1]} in {best_pair[0]} -- ₹{top_revenue:,.0f} from {order_num} orders")

Best: Aarav Shah in North -- ₹162,295 from 6 orders


#### Regional Insight: The West region outperforme others with total revenue of ₹200,275 driven primarily by highest volume of orders we should analyze their regional marketing tactics to improve performance in South.

#### Salesperson Insight: Aarav Shah is the top performer largely due to his dominance in Electronics category which has the highest revenue per unit he should lead a training session for the team on selling high-value technical products.

#### Time Trend Insight:Revenue peaked sharply in Month 3 with a massive 100.45% growth, but has since faced a consistent decline for four consecutive months (Months 4–7). This suggests that while Month 3 was a major success, the business has lost momentum and needs a new strategy to stop the ongoing downward trend.
